
# IMTalker - Generator Training: Video Preprocessing (notebook 1/3)

This notebook prepares the **video-side data** required for **Generator Training**
(README: *Train -> 2. Train the generator -> Data Preparation*).

It performs four steps:

| Step | Output | Format |
|------|--------|--------|
| 1. Crop raw videos to 512x512 @ 25 fps (face-centered) | `videos_512/<stem>.mp4` | intermediate |
| 2. Motion latents (renderer motion encoder) | `generator_dataset/motion/<stem>.pt` | `Tensor[T, 32]` |
| 3. SMIRK 6D pose parameters | `generator_dataset/smirk/<stem>.pt` | `dict` with `pose_params (T,3)`, `cam (T,3)`, ... |
| 4. Gaze directions (L2CS-Net) | `generator_dataset/gaze/<stem>.npy` | `ndarray[T, 2]` (yaw, pitch in radians) |

Afterwards run `02_preprocess_audio.ipynb`, then `03_verify_dataset.ipynb`.

**Repository code this notebook follows:**
- `README.md` - generator training data preparation (directory layout, tools)
- `generator/dataset.py` (`AudioMotionSmirkGazeDataset`) - the exact files / shapes / dtypes that training reads
- `app.py` (`crop_video_stable`) - 512x512 face crop (implementation of [talkingfaceprocess](https://github.com/liutaocode/talking_face_preprocessing))
- `renderer/models.py` (`IMTRenderer.mot_encode`) + `renderer/inference.py` - motion latents
- [SMIRK](https://github.com/georgeretsi/smirk) `demo.py` / `utils/mediapipe_utils.py` - pose / expression extraction
- [L2CS-Net](https://github.com/Ahmednull/L2CS-Net) `l2cs/pipeline.py` - gaze extraction



## 0. Prerequisites

1. **Kernel**: select the Python environment you use for IMTalker (`requirement.txt` installed) as the Jupyter kernel.
2. **ffmpeg** must be on PATH (used for video cropping, exactly like `app.py`).
3. **Input**: one folder with your raw `.mp4` talking-face videos (one face per video).
4. **GPU**: CUDA recommended (all steps also run on CPU, but much slower).
5. Videos with fewer than **60 frames at 25 fps (2.4 s)** cannot be used by the generator dataset
   (`required_len = wav2vec_sec * fps + num_prev_frames = 50 + 10` in `generator/dataset.py`); they are reported.
6. Every step is **resumable**: outputs that already exist are skipped unless `OVERWRITE = True`.
7. If your videos are **already cropped to 512x512** with the face filling the frame, point
   `CROPPED_VIDEO_DIR` at that folder and skip Step 1.


In [ ]:
!git clone https://github.com/bigai-nlco/IMTalker.git

In [ ]:

# ============================================================================
# CONFIGURATION - EDIT THE PATHS BELOW
# ============================================================================
from pathlib import Path

# --- Input: folder with your raw .mp4 files ---------------------------------
RAW_VIDEO_DIR = Path(r"/workspace/IMTalker/assets/video")   # <-- CHANGE ME

# --- Repository root (folder that contains app.py, generator/, renderer/) ---
REPO_ROOT = Path(r"/workspace/IMTalker")

# --- Output work dir (created automatically) --------------------------------
WORK_DIR = REPO_ROOT / "preprocessing_output"
CROPPED_VIDEO_DIR = WORK_DIR / "videos_512"          # step 1 output
DATASET_DIR = WORK_DIR / "generator_dataset"         # final dataset root (README layout)
MOTION_DIR = DATASET_DIR / "motion"                  # step 2 output
SMIRK_DIR = DATASET_DIR / "smirk"                    # step 3 output
GAZE_DIR = DATASET_DIR / "gaze"                      # step 4 output
CKPT_DIR = WORK_DIR / "checkpoints"                  # downloaded models / checkpoints

# --- Device -------------------------------------------------------------------
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Video cropping parameters (defaults from app.py / talkingfaceprocess) -----
EXPANDED_RATIO = 0.6     # bbox expansion ratio (app.py crop_video_stable)
SKIP_PER_FRAME = 15      # run face detection every N frames (app.py)
TARGET_SIZE = 512        # output resolution (README)
TARGET_FPS = 25          # IMTalker fps (generator/options/base_options.py --fps 25)

# --- Batch sizes (tune to your GPU) -------------------------------------------
BATCH_SIZE_MOTION = 16
BATCH_SIZE_SMIRK = 32
BATCH_SIZE_GAZE = 32

OVERWRITE = False  # True -> recompute even if the output file already exists

# --- Sanity checks -------------------------------------------------------------
assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"
assert (REPO_ROOT / "generator" / "dataset.py").exists(), "generator/dataset.py not found under REPO_ROOT"
assert RAW_VIDEO_DIR.exists(), f"RAW_VIDEO_DIR not found: {RAW_VIDEO_DIR} - edit the config above!"

for d in [CROPPED_VIDEO_DIR, MOTION_DIR, SMIRK_DIR, GAZE_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

videos = sorted(p for p in RAW_VIDEO_DIR.iterdir()
                if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv", ".webm"})
assert videos, f"no video files found in {RAW_VIDEO_DIR}"
print(f"Found {len(videos)} raw videos in {RAW_VIDEO_DIR}")
for v in videos[:10]:
    print("  ", v.name)
if len(videos) > 10:
    print("   ...")


In [ ]:

# Check ffmpeg (required for video cropping, same as app.py)
import subprocess, shutil

if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "ffmpeg not found on PATH. Install it (e.g. 'winget install ffmpeg' or from https://ffmpeg.org) "
        "and restart the kernel.")
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])


In [ ]:
# Dependencies for this notebook (run once; safe to re-run).
# Keep the kernel in the environment you use for IMTalker.

# Repo dependencies from requirement.txt that this notebook needs.
# NOTE: requirement.txt pins transformers==4.30.2, but that version CANNOT be installed on
# Python 3.12 (its old tokenizers dependency has no prebuilt cp312 wheel and would need a
# Rust compiler). transformers >= 4.38 works identically for the repo's generator/wav2vec2.py.
%pip install -q "face_alignment==1.4.1" "timm==1.0.9" librosa "transformers>=4.38,<4.42"

# Extra dependencies that are NOT in requirement.txt
%pip install -q mediapipe scikit-image gdown "huggingface_hub>=0.19.3,<0.26"
%pip install -q git+https://github.com/elliottzheng/face-detection

# NOTE: 'mediapipe' pulls in opencv-contrib-python which may overwrite the repo-pinned
# cv2 (opencv-python<4.10). If you later hit cv2 errors, restore it with:
#   %pip install --force-reinstall "opencv-python<4.10.0.0"


In [ ]:

import os, sys, gc, shutil, subprocess, urllib.request
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import torchvision.transforms as transforms

# Make the repo modules importable (same as app.py does)
sys.path.insert(0, str(REPO_ROOT))

import face_alignment  # repo dependency (requirement.txt)

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("cv2", cv2.__version__, "| numpy", np.__version__)
print("DEVICE:", DEVICE)


In [ ]:

# Download all required checkpoints (each is skipped if already present).
#  - renderer.ckpt          (IMTalker, HuggingFace cbsjtu01/IMTalker)   ~2.1 GB
#  - SMIRK_em1.pt           (SMIRK,    gdrive id 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE)
#  - face_landmarker.task   (MediaPipe landmarker model)
#  - L2CSNet_gaze360.pkl    (L2CS-Net, gdrive folder 17p6ORr-JQJcw-eYtG2WGNiuS_qVKwdWd)

RENDERER_CKPT = CKPT_DIR / "renderer.ckpt"
SMIRK_CKPT = CKPT_DIR / "SMIRK_em1.pt"
FACE_LANDMARKER = CKPT_DIR / "face_landmarker.task"
L2CS_CKPT = CKPT_DIR / "L2CSNet_gaze360.pkl"

def _download_renderer():
    if RENDERER_CKPT.exists() and RENDERER_CKPT.stat().st_size > 1_000_000:
        print("[skip] renderer.ckpt already present"); return
    from huggingface_hub import hf_hub_download
    print("[download] renderer.ckpt (~2.1 GB) from cbsjtu01/IMTalker ...")
    hf_hub_download(repo_id="cbsjtu01/IMTalker", filename="renderer.ckpt", local_dir=str(CKPT_DIR))

def _download_smirk():
    if SMIRK_CKPT.exists() and SMIRK_CKPT.stat().st_size > 1_000_000:
        print("[skip] SMIRK_em1.pt already present"); return
    import gdown
    print("[download] SMIRK_em1.pt from Google Drive ...")
    gdown.download(id="1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE", output=str(SMIRK_CKPT), quiet=False)
    assert SMIRK_CKPT.exists(), "SMIRK download failed - download SMIRK_em1.pt manually (see SMIRK GitHub README)"

def _download_landmarker():
    if FACE_LANDMARKER.exists() and FACE_LANDMARKER.stat().st_size > 100_000:
        print("[skip] face_landmarker.task already present"); return
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
    print("[download] MediaPipe face_landmarker.task ...")
    urllib.request.urlretrieve(url, FACE_LANDMARKER)

def _download_l2cs():
    if L2CS_CKPT.exists() and L2CS_CKPT.stat().st_size > 1_000_000:
        print("[skip] L2CSNet_gaze360.pkl already present"); return
    # Primary: HuggingFace mirror (reliable - no Google Drive folder scraping)
    try:
        from huggingface_hub import hf_hub_download
        print("[download] L2CSNet_gaze360.pkl from HuggingFace mirror tianfxc/l2cs ...")
        hf_hub_download(repo_id="tianfxc/l2cs", filename="L2CSNet_gaze360.pkl", local_dir=str(CKPT_DIR))
    except Exception as e:
        print("HuggingFace download failed:", e)
    # Fallback: Google Drive folder via gdown
    if not L2CS_CKPT.exists():
        try:
            import gdown
            print("[download] L2CSNet_gaze360.pkl from Google Drive folder ...")
            gdown.download_folder(
                url="https://drive.google.com/drive/folders/17p6ORr-JQJcw-eYtG2WGNiuS_qVKwdWd",
                output=str(CKPT_DIR / "_l2cs_folder"), quiet=False, use_cookies=False)
            for p in (CKPT_DIR / "_l2cs_folder").rglob("*.pkl"):
                if "gaze360" in p.name.lower():
                    shutil.move(str(p), L2CS_CKPT); break
        except Exception as e:
            print("gdown folder download failed:", e)
    if not L2CS_CKPT.exists():
        raise RuntimeError(
            "Could not download L2CSNet_gaze360.pkl automatically.\n"
            "Download it manually from one of:\n"
            "  - https://huggingface.co/tianfxc/l2cs (file L2CSNet_gaze360.pkl)\n"
            "  - https://drive.google.com/drive/folders/17p6ORr-JQJcw-eYtG2WGNiuS_qVKwdWd\n"
            "and place it at:\n  " + str(L2CS_CKPT))
    # sanity check: a ResNet50 state_dict is tens of MB
    assert L2CS_CKPT.stat().st_size > 10_000_000, "L2CSNet_gaze360.pkl looks too small - download may be corrupt"

_download_renderer(); _download_smirk(); _download_landmarker(); _download_l2cs()
print("All checkpoints ready.")



## Step 1 - Crop videos to 512x512 @ 25 fps

Follows the README instruction (*"crop the raw video data into 512x512 resolution videos where the face occupies the main region"*)
and reproduces `app.py: crop_video_stable()` (which implements [talkingfaceprocess](https://github.com/liutaocode/talking_face_preprocessing) `extract_cropped_faces.py`):

1. Detect the face every `SKIP_PER_FRAME` frames with the `face_alignment` detector (frames resized to height 360 for speed, exactly like `app.py`).
2. Take the **median** bbox center / size over the whole video (stable crop, no jitter).
3. Expand the box by `EXPANDED_RATIO = 0.6`, make it square, clamp to the frame.
4. `ffmpeg` crop + `scale=512:512` (lanczos) + **`-r 25`**.

`-r 25` standardizes the frame rate to the 25 fps that IMTalker hard-codes (`generator/options/base_options.py: --fps 25`);
this guarantees motion / smirk / gaze / audio features stay frame-aligned.


In [ ]:

# Face detector used by app.py (face_alignment's internal detector, on CPU like app.py)
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, device="cpu", flip_input=False)

def crop_video_stable(src: Path, dst: Path, expanded_ratio=EXPANDED_RATIO, skip_per_frame=SKIP_PER_FRAME):
    # --- port of app.py crop_video_stable (which follows talkingfaceprocess) ---
    video = cv2.VideoCapture(str(src))
    index = 0
    bboxes_lists = []
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width == 0 or height == 0:
        video.release()
        raise RuntimeError("cannot read video (0x0 frames)")
    while video.isOpened():
        success = video.grab()
        if not success:
            break
        if index % skip_per_frame == 0:
            success, frame = video.retrieve()
            if not success:
                break
            h, w = frame.shape[:2]
            mult = 360.0 / h
            resized_frame = cv2.resize(frame, dsize=(0, 0), fx=mult, fy=mult,
                                       interpolation=cv2.INTER_AREA if mult < 1 else cv2.INTER_CUBIC)
            try:
                detected_bboxes = fa.face_detector.detect_from_image(resized_frame)
            except Exception:
                detected_bboxes = None
            current_frame_bboxes = []
            if detected_bboxes is not None:
                for d_box in detected_bboxes:
                    bx1, by1, bx2, by2, score = d_box
                    if score > 0.5:
                        current_frame_bboxes.append([int(bx1 / mult), int(by1 / mult), int(bx2 / mult), int(by2 / mult), score])
            if len(current_frame_bboxes) > 0:
                max_bboxes = max(current_frame_bboxes, key=lambda bbox: bbox[2] - bbox[0])
                bboxes_lists.append(max_bboxes)
        index += 1
    video.release()

    x_center_lists, y_center_lists, width_lists, height_lists = [], [], [], []
    for bbox in bboxes_lists:
        x1, y1, x2, y2 = bbox[:4]
        x_center_lists.append((x1 + x2) / 2); y_center_lists.append((y1 + y2) / 2)
        width_lists.append(x2 - x1); height_lists.append(y2 - y1)

    if not x_center_lists:
        # no face found at all -> copy the original file, exactly like app.py (downstream steps will report it)
        print(f"   [warn] no face detected in {src.name} - copying original (app.py fallback)")
        shutil.copy(str(src), str(dst))
        return

    x_center = sorted(x_center_lists)[len(x_center_lists) // 2]
    y_center = sorted(y_center_lists)[len(y_center_lists) // 2]
    median_width = sorted(width_lists)[len(width_lists) // 2]
    median_height = sorted(height_lists)[len(height_lists) // 2]

    expanded_width = int(median_width * (1 + expanded_ratio))
    expanded_height = int(median_height * (1 + expanded_ratio))
    fixed_cropped_width = min(max(expanded_width, expanded_height), width, height)
    x1, y1 = int(x_center - fixed_cropped_width / 2), int(y_center - fixed_cropped_width / 2)
    x1 = max(0, x1); y1 = max(0, y1)
    if x1 + fixed_cropped_width > width: x1 = width - fixed_cropped_width
    if y1 + fixed_cropped_width > height: y1 = height - fixed_cropped_width

    cmd = (f'ffmpeg -i "{src}" -filter:v "crop={fixed_cropped_width}:{fixed_cropped_width}:{x1}:{y1},'
           f'scale={TARGET_SIZE}:{TARGET_SIZE}:flags=lanczos" -r {TARGET_FPS} '
           f'-c:v libx264 -crf 18 -preset slow -c:a aac -b:a 128k "{dst}" -y -loglevel error')
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError("ffmpeg failed")

crop_failures = []
todo = [v for v in videos if OVERWRITE or not (CROPPED_VIDEO_DIR / (v.stem + ".mp4")).exists()]
print(f"Cropping {len(todo)}/{len(videos)} videos ...")
for v in tqdm(todo, desc="crop 512x512"):
    dst = CROPPED_VIDEO_DIR / (v.stem + ".mp4")
    try:
        crop_video_stable(v, dst)
    except Exception as e:
        print(f"[FAIL] {v.name}: {e}")
        crop_failures.append(v.name)

cropped_videos = sorted(CROPPED_VIDEO_DIR.glob("*.mp4"))
print(f"Done. {len(cropped_videos)} cropped videos in {CROPPED_VIDEO_DIR} (failures: {len(crop_failures)})")



## Step 2 - Motion latents (`motion/*.pt`)

README: *"motion latents obtained by feeding each video frame into the motion encoder in the renderer"*.

- Model: `IMTRenderer.latent_token_encoder` (== `mot_encode`, `renderer/models.py`), weights from the official
  `renderer.ckpt` (HuggingFace `cbsjtu01/IMTalker`), loaded exactly like `app.py: InferenceAgent._load_ckpt`
  (strip the `"gen."` prefix, `strict=False`).
- Input per frame: 512x512 **RGB** image in `[0,1]` - identical to `renderer/dataset.py: TFDataset` /
  `renderer/inference.py` (`transforms.Resize((512,512)) + ToTensor()`).
- Output per frame: 32-dim latent (`--dim_motion 32`), stacked over time -> `Tensor[T, 32]`, saved with `torch.save`.

Frames are processed in batches (the encoder has no batch-dependent layers, so batching is exact).


In [ ]:

import argparse
from renderer.models import IMTRenderer

# Constructor args used by app.py AppConfig / renderer/train.py defaults
renderer_args = argparse.Namespace(
    swin_res_threshold=128,
    num_heads=8,
    window_size=8,
)
renderer = IMTRenderer(renderer_args)

def torch_load(path):
    # weights_only=False: these are official checkpoints containing pickled state dicts
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:  # older torch without the kwarg
        return torch.load(str(path), map_location="cpu")

# --- load checkpoint exactly like app.py InferenceAgent._load_ckpt ---
ckpt = torch_load(RENDERER_CKPT)["state_dict"]
clean_state_dict = {k.replace("gen.", ""): v for k, v in ckpt.items() if k.startswith("gen.")}
msg = renderer.load_state_dict(clean_state_dict, strict=False)
renderer.to(DEVICE).eval()
print(f"Loaded {len(clean_state_dict)} 'gen.*' weights | missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
assert len(clean_state_dict) > 100, "renderer.ckpt looks wrong (no 'gen.*' keys found)"


In [ ]:

motion_transform = transforms.Compose([transforms.Resize((512, 512)), transforms.ToTensor()])

@torch.no_grad()
def extract_motion(video_path: Path, out_path: Path, batch_size=BATCH_SIZE_MOTION):
    cap = cv2.VideoCapture(str(video_path))
    latents, buf = [], []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)          # renderer expects RGB
        t = motion_transform(Image.fromarray(frame_rgb))             # 512x512, [0,1]
        buf.append(t)
        if len(buf) == batch_size:
            latents.append(renderer.mot_encode(torch.stack(buf).to(DEVICE)).cpu())
            buf = []
    if buf:
        latents.append(renderer.mot_encode(torch.stack(buf).to(DEVICE)).cpu())
    cap.release()
    motion = torch.cat(latents, dim=0)     # (T, 32)
    assert motion.ndim == 2 and motion.shape[1] == 32, f"unexpected motion shape {motion.shape}"
    torch.save(motion, str(out_path))
    return motion.shape[0]

motion_failures, motion_frames = [], {}
todo = [v for v in cropped_videos if OVERWRITE or not (MOTION_DIR / (v.stem + ".pt")).exists()]
print(f"Extracting motion latents for {len(todo)}/{len(cropped_videos)} videos ...")
for v in tqdm(todo, desc="motion"):
    try:
        motion_frames[v.stem] = extract_motion(v, MOTION_DIR / (v.stem + ".pt"))
    except Exception as e:
        print(f"[FAIL] {v.name}: {e}")
        motion_failures.append(v.name)

# videos too short for the generator dataset (required_len = 2*25 + 10 = 60)
too_short = {k: n for k, n in motion_frames.items() if n < 60}
if too_short:
    print(f"[warn] {len(too_short)} videos shorter than 60 frames (unusable for training):", list(too_short.items())[:5])

del renderer
gc.collect()
torch.cuda.empty_cache()
print(f"motion/: {len(list(MOTION_DIR.glob('*.pt')))} files")



## Step 3 - SMIRK parameters (`smirk/*.pt`)

README: *"6D pose parameters for each frame extracted with SMIRK"*.

`generator/dataset.py: load_pose()` reads `smirk["pose_params"]` `(T,3)` and `smirk["cam"]` `(T,3)`
(the 6D pose = 3 global rotation + 3 camera). We reproduce the **official SMIRK inference pipeline**
(`demo.py` of [georgeretsi/smirk](https://github.com/georgeretsi/smirk)) exactly:

1. MediaPipe **FaceLandmarker** (478 landmarks, `min_face_detection_confidence=0.1`) - `utils/mediapipe_utils.py`
2. `crop_face(..., scale=1.4, image_size=224)` similarity transform - `demo.py`
3. `skimage.transform.warp` to 224x224, `cv2.cvtColor(BGR2RGB)`, `/255.0` - `demo.py`
4. `SmirkEncoder` (mobilenetv3 backbones) with the official `SMIRK_em1.pt` checkpoint (keys `smirk_encoder.*`)

Saved dict (all `float32`; the first two are what training reads):
`pose_params (T,3)`, `cam (T,3)`, `shape_params (T,300)`, `expression_params (T,50)`, `eyelid_params (T,2)`, `jaw_params (T,3)`.

Frames where MediaPipe finds no face reuse the nearest valid frame's parameters (rare on cropped talking-face
videos); a video with *no* valid frame is skipped with an error.


## Step 3a - System libraries for MediaPipe (Linux / headless servers only)

MediaPipe's native library links against OpenGL/EGL. On a **headless Linux server** these
system libraries are usually missing, which causes:

    OSError: libEGL.so.1: cannot open shared object file: No such file or directory

This is a **system** dependency, not a Python package, so `pip` cannot install it.
Run the cell below to install it with `apt-get` (Debian/Ubuntu) - it needs **root or sudo**.
Skip it on Windows / macOS / machines with a desktop environment.

In [ ]:
import platform, shutil, subprocess, os, ctypes

def install_system_gl_libs():
    if platform.system() != "Linux":
        print("Not Linux - skipping system GL library install.")
        return
    # GL / X11 libraries that mediapipe (and opencv) need on a headless server
    pkgs = ["libegl1", "libgl1", "libgles2", "libglib2.0-0",
            "libsm6", "libxext6", "libxrender1", "libfontconfig1"]
    sudo = shutil.which("sudo")
    is_root = hasattr(os, "geteuid") and os.geteuid() == 0
    if not is_root and sudo is None:
        print("[MANUAL ACTION NEEDED] No root and no sudo available.")
        print("Ask your administrator to run:")
        print("  apt-get update && apt-get install -y " + " ".join(pkgs))
        print("(on RHEL/CentOS:  yum install -y mesa-libEGL mesa-libGL glib2)")
        return
    # 'sudo -n' = non-interactive: fails fast instead of hanging on a password prompt
    prefix = [] if is_root else [sudo, "-n"]
    for cmd in (["apt-get", "update"], ["apt-get", "install", "-y"] + pkgs, ["ldconfig"]):
        full = prefix + cmd
        print("Running:", " ".join(full))
        subprocess.run(full, check=False)
    # verify the library now loads
    try:
        ctypes.CDLL("libEGL.so.1")
        print("libEGL.so.1 now loads OK - re-run the SMIRK setup cell below.")
    except OSError as e:
        print("libEGL.so.1 still not found:", e)
        print("Restart the kernel and re-run, or install the packages manually (see message above).")

install_system_gl_libs()


In [ ]:

# --- SmirkEncoder: verbatim from SMIRK src/smirk_encoder.py -------------------
# (only change: pretrained=False - all weights come from SMIRK_em1.pt anyway)
import timm

def create_backbone(backbone_name, pretrained=True):
    backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=True)
    feature_dim = backbone.feature_info[-1]["num_chs"]
    return backbone, feature_dim

class PoseEncoder(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder, feature_dim = create_backbone("tf_mobilenetv3_small_minimal_100", pretrained=False)
        self.pose_cam_layers = torch.nn.Sequential(torch.nn.Linear(feature_dim, 6))
        self.init_weights()
    def init_weights(self):
        self.pose_cam_layers[-1].weight.data *= 0.001
        self.pose_cam_layers[-1].bias.data *= 0.001
        self.pose_cam_layers[-1].weight.data[3] = 0
        self.pose_cam_layers[-1].bias.data[3] = 7
    def forward(self, img):
        features = self.encoder(img)[-1]
        features = F.adaptive_avg_pool2d(features, (1, 1)).squeeze(-1).squeeze(-1)
        outputs = {}
        pose_cam = self.pose_cam_layers(features).reshape(img.size(0), -1)
        outputs["pose_params"] = pose_cam[..., :3]
        outputs["cam"] = pose_cam[..., 3:]
        return outputs

class ShapeEncoder(torch.nn.Module):
    def __init__(self, n_shape=300) -> None:
        super().__init__()
        self.encoder, feature_dim = create_backbone("tf_mobilenetv3_large_minimal_100", pretrained=False)
        self.shape_layers = torch.nn.Sequential(torch.nn.Linear(feature_dim, n_shape))
        self.init_weights()
    def init_weights(self):
        self.shape_layers[-1].weight.data *= 0
        self.shape_layers[-1].bias.data *= 0
    def forward(self, img):
        features = self.encoder(img)[-1]
        features = F.adaptive_avg_pool2d(features, (1, 1)).squeeze(-1).squeeze(-1)
        parameters = self.shape_layers(features).reshape(img.size(0), -1)
        return {"shape_params": parameters}

class ExpressionEncoder(torch.nn.Module):
    def __init__(self, n_exp=50) -> None:
        super().__init__()
        self.encoder, feature_dim = create_backbone("tf_mobilenetv3_large_minimal_100", pretrained=False)
        self.expression_layers = torch.nn.Sequential(torch.nn.Linear(feature_dim, n_exp + 2 + 3))
        self.n_exp = n_exp
        self.init_weights()
    def init_weights(self):
        self.expression_layers[-1].weight.data *= 0.1
        self.expression_layers[-1].bias.data *= 0.1
    def forward(self, img):
        features = self.encoder(img)[-1]
        features = F.adaptive_avg_pool2d(features, (1, 1)).squeeze(-1).squeeze(-1)
        parameters = self.expression_layers(features).reshape(img.size(0), -1)
        outputs = {}
        outputs["expression_params"] = parameters[..., :self.n_exp]
        outputs["eyelid_params"] = torch.clamp(parameters[..., self.n_exp:self.n_exp + 2], 0, 1)
        outputs["jaw_params"] = torch.cat(
            [F.relu(parameters[..., self.n_exp + 2].unsqueeze(-1)),
             torch.clamp(parameters[..., self.n_exp + 3:self.n_exp + 5], -.2, .2)], dim=-1)
        return outputs

class SmirkEncoder(torch.nn.Module):
    def __init__(self, n_exp=50, n_shape=300) -> None:
        super().__init__()
        self.pose_encoder = PoseEncoder()
        self.shape_encoder = ShapeEncoder(n_shape=n_shape)
        self.expression_encoder = ExpressionEncoder(n_exp=n_exp)
    def forward(self, img):
        outputs = {}
        outputs.update(self.pose_encoder(img))
        outputs.update(self.shape_encoder(img))
        outputs.update(self.expression_encoder(img))
        return outputs

# --- checkpoint loading: verbatim from SMIRK demo.py --------------------------
smirk_encoder = SmirkEncoder().to(DEVICE)
smirk_ckpt = torch_load(SMIRK_CKPT)
checkpoint_encoder = {k.replace("smirk_encoder.", ""): v for k, v in smirk_ckpt.items() if "smirk_encoder" in k}
smirk_encoder.load_state_dict(checkpoint_encoder)
smirk_encoder.eval()
print(f"SMIRK encoder loaded ({len(checkpoint_encoder)} tensors)")

# --- MediaPipe landmarker: verbatim from SMIRK utils/mediapipe_utils.py -------
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

base_options = mp_python.BaseOptions(model_asset_path=str(FACE_LANDMARKER))
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1,
    min_face_detection_confidence=0.1,
    min_face_presence_confidence=0.1,
)
landmarker = mp_vision.FaceLandmarker.create_from_options(options)

def run_mediapipe(image_bgr):
    image_numpy = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_numpy)
    detection_result = landmarker.detect(image)
    if len(detection_result.face_landmarks) == 0:
        return None
    face_landmarks = detection_result.face_landmarks[0]
    face_landmarks_numpy = np.zeros((478, 3))
    for i, landmark in enumerate(face_landmarks):
        face_landmarks_numpy[i] = [landmark.x * image.width, landmark.y * image.height, landmark.z]
    return face_landmarks_numpy

# --- crop_face: verbatim from SMIRK demo.py -----------------------------------
from skimage.transform import estimate_transform, warp

def crop_face(frame, landmarks, scale=1.0, image_size=224):
    left = np.min(landmarks[:, 0]); right = np.max(landmarks[:, 0])
    top = np.min(landmarks[:, 1]); bottom = np.max(landmarks[:, 1])
    old_size = (right - left + bottom - top) / 2
    center = np.array([right - (right - left) / 2.0, bottom - (bottom - top) / 2.0])
    size = int(old_size * scale)
    src_pts = np.array([[center[0] - size / 2, center[1] - size / 2],
                        [center[0] - size / 2, center[1] + size / 2],
                        [center[0] + size / 2, center[1] - size / 2]])
    DST_PTS = np.array([[0, 0], [0, image_size - 1], [image_size - 1, 0]])
    tform = estimate_transform("similarity", src_pts, DST_PTS)
    return tform


In [ ]:

SMIRK_KEYS = ["pose_params", "cam", "shape_params", "expression_params", "eyelid_params", "jaw_params"]

@torch.no_grad()
def extract_smirk(video_path: Path, out_path: Path, batch_size=BATCH_SIZE_SMIRK):
    cap = cv2.VideoCapture(str(video_path))
    feats = {k: [] for k in SMIRK_KEYS}
    valid_idx, buf = [], []
    n_frames = 0

    def flush():
        if not buf:
            return
        out = smirk_encoder(torch.stack(buf).to(DEVICE))
        for k in SMIRK_KEYS:
            feats[k].append(out[k].cpu())
        buf.clear()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        kpt = run_mediapipe(frame)     # (478, 3) or None  [SMIRK mediapipe_utils.py]
        if kpt is not None:
            tform = crop_face(frame, kpt[..., :2], scale=1.4, image_size=224)   # SMIRK demo.py
            cropped = warp(frame, tform.inverse, output_shape=(224, 224), preserve_range=True).astype(np.uint8)
            cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)                 # SMIRK demo.py
            cropped = cv2.resize(cropped, (224, 224))
            buf.append(torch.tensor(cropped).permute(2, 0, 1).float() / 255.0)
            valid_idx.append(n_frames)
            if len(buf) == batch_size:
                flush()
        n_frames += 1
    flush()
    cap.release()
    if not valid_idx:
        raise RuntimeError("no face detected in any frame")

    for k in SMIRK_KEYS:
        feats[k] = torch.cat(feats[k], dim=0)          # (n_valid, D)

    # fill frames without detection (forward + backward fill from nearest valid frame)
    mask = np.zeros(n_frames, dtype=bool)
    mask[valid_idx] = True
    data = {}
    for k in SMIRK_KEYS:
        arr = torch.zeros((n_frames,) + tuple(feats[k].shape[1:]), dtype=torch.float32)
        arr[mask] = feats[k]
        last = None
        for t in range(n_frames):                      # forward fill
            if mask[t]: last = t
            elif last is not None: arr[t] = arr[last]
        last = None
        for t in range(n_frames - 1, -1, -1):          # backward fill
            if mask[t]: last = t
            elif last is not None: arr[t] = arr[last]
        data[k] = arr
    torch.save(data, str(out_path))
    return n_frames, len(valid_idx)

smirk_failures = []
todo = [v for v in cropped_videos if OVERWRITE or not (SMIRK_DIR / (v.stem + ".pt")).exists()]
print(f"Extracting SMIRK params for {len(todo)}/{len(cropped_videos)} videos ...")
for v in tqdm(todo, desc="smirk"):
    try:
        n, nv = extract_smirk(v, SMIRK_DIR / (v.stem + ".pt"))
        if nv < n:
            print(f"   [warn] {v.name}: {n - nv}/{n} frames had no face - filled from nearest valid frame")
    except Exception as e:
        print(f"[FAIL] {v.name}: {e}")
        smirk_failures.append(v.name)

del smirk_encoder
gc.collect()
torch.cuda.empty_cache()
print(f"smirk/: {len(list(SMIRK_DIR.glob('*.pt')))} files")



## Step 4 - Gaze (`gaze/*.npy`)

README: *"gaze directions extracted using L2CS-Net"*.

Reproduces the **official L2CS-Net pipeline** ([l2cs/pipeline.py](https://github.com/Ahmednull/L2CS-Net/blob/main/l2cs/pipeline.py)):

1. RetinaFace (`face_detection` package) detects the face (confidence > 0.5, largest box if several).
2. The face bbox crop is resized to 224x224 RGB.
3. L2CS transforms: `Resize(448)` + ImageNet normalization (`l2cs/utils.py`).
4. `L2CSNet_gaze360.pkl` (ResNet50, 90 bins) -> softmax expectation -> `*4 - 180` degrees -> radians.

Output: `ndarray[T, 2]`, **`[:,0] = yaw`, `[:,1] = pitch`** (radians).

> **Convention note:** the repository does not prescribe the (yaw, pitch) ordering - `generator/FM.py` only
> projects the 2-D gaze vector through a learned `Linear(2, dim_c)`. This notebook fixes the convention
> `[yaw, pitch]`; keep it consistent if you ever extract gaze for inference (`generator/generate.py --gaze_path`).


In [ ]:

# --- L2CS-Net model: verbatim from l2cs/model.py + utils.getArch --------------
import math
import torchvision

class L2CS(torch.nn.Module):
    def __init__(self, block, layers, num_bins):
        self.inplanes = 64
        super(L2CS, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(64)
        self.relu = torch.nn.ReLU(inplace=True)
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.fc_yaw_gaze = torch.nn.Linear(512 * block.expansion, num_bins)
        self.fc_pitch_gaze = torch.nn.Linear(512 * block.expansion, num_bins)
        self.fc_finetune = torch.nn.Linear(512 * block.expansion + 3, 3)
        for m in self.modules():
            if isinstance(m, torch.nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2.0 / n))
            elif isinstance(m, torch.nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = torch.nn.Sequential(
                torch.nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                torch.nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return torch.nn.Sequential(*layers)
    def forward(self, x):
        x = self.conv1(x); x = self.bn1(x); x = self.relu(x); x = self.maxpool(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        pre_yaw_gaze = self.fc_yaw_gaze(x)
        pre_pitch_gaze = self.fc_pitch_gaze(x)
        return pre_yaw_gaze, pre_pitch_gaze

def getArch(arch, bins):  # from l2cs/utils.py (only ResNet50 is needed here)
    assert arch == "ResNet50"
    return L2CS(torchvision.models.resnet.Bottleneck, [3, 4, 6, 3], bins)

l2cs_model = getArch("ResNet50", 90)
l2cs_model.load_state_dict(torch_load(L2CS_CKPT))   # L2CSNet_gaze360.pkl
l2cs_model.to(DEVICE).eval()
print("L2CS model loaded")

# --- transforms / softmax: verbatim from l2cs/utils.py + pipeline.py ----------
from torchvision import transforms as tv_transforms
l2cs_transform = tv_transforms.Compose([
    tv_transforms.ToPILImage(),
    tv_transforms.Resize(448),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
l2cs_softmax = torch.nn.Softmax(dim=1)
idx_tensor = torch.FloatTensor([idx for idx in range(90)]).to(DEVICE)

from face_detection import RetinaFace
retina = RetinaFace(gpu_id=0) if DEVICE.type == "cuda" else RetinaFace()

@torch.no_grad()
def predict_gaze(face_imgs_rgb):
    # face_imgs_rgb: (N, 224, 224, 3) uint8 RGB face crops (pipeline.py)
    imgs = torch.stack([l2cs_transform(im) for im in face_imgs_rgb]).to(DEVICE)
    # NOTE (L2CS quirk): the FIRST output is the pitch head and the SECOND the yaw head -
    # exactly how l2cs/pipeline.py and L2CS-Net train.py unpack the model outputs.
    gaze_pitch, gaze_yaw = l2cs_model(imgs)
    pitch_predicted = torch.sum(l2cs_softmax(gaze_pitch) * idx_tensor, dim=1) * 4 - 180
    yaw_predicted = torch.sum(l2cs_softmax(gaze_yaw) * idx_tensor, dim=1) * 4 - 180
    pitch = pitch_predicted.cpu().numpy() * np.pi / 180.0
    yaw = yaw_predicted.cpu().numpy() * np.pi / 180.0
    return yaw, pitch


In [ ]:

@torch.no_grad()
def extract_gaze(video_path: Path, out_path: Path, batch_size=BATCH_SIZE_GAZE):
    cap = cv2.VideoCapture(str(video_path))
    yaws, pitches, valid_idx, buf = [], [], [], []
    n_frames = 0

    def flush():
        if not buf:
            return
        yw, pt = predict_gaze(np.stack(buf))
        yaws.append(yw); pitches.append(pt)
        buf.clear()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        try:
            faces = retina(frame)   # list of (box, landmarks, score)
        except Exception:
            faces = None
        best = None
        if faces:
            for box, landmark, score in faces:      # l2cs pipeline.py
                if score < 0.5:
                    continue
                if best is None or (box[2] - box[0]) > (best[0][2] - best[0][0]):
                    best = (box, landmark, score)
        if best is not None:
            box = best[0]
            x_min = max(int(box[0]), 0); y_min = max(int(box[1]), 0)
            x_max = int(box[2]); y_max = int(box[3])
            img = frame[y_min:y_max, x_min:x_max]
            if img.size > 0:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
                buf.append(img)
                valid_idx.append(n_frames)
                if len(buf) == batch_size:
                    flush()
        n_frames += 1
    flush()
    cap.release()
    if not valid_idx:
        raise RuntimeError("no face detected in any frame")

    yaw = np.concatenate(yaws); pitch = np.concatenate(pitches)

    # fill frames without detection (forward + backward fill from nearest valid frame)
    yaw_full = np.zeros(n_frames, dtype=np.float32); pitch_full = np.zeros(n_frames, dtype=np.float32)
    yaw_full[valid_idx] = yaw; pitch_full[valid_idx] = pitch
    mask = np.zeros(n_frames, dtype=bool); mask[valid_idx] = True
    for arr in (yaw_full, pitch_full):
        last = None
        for t in range(n_frames):
            if mask[t]: last = t
            elif last is not None: arr[t] = arr[last]
        last = None
        for t in range(n_frames - 1, -1, -1):
            if mask[t]: last = t
            elif last is not None: arr[t] = arr[last]

    gaze = np.stack([yaw_full, pitch_full], axis=1).astype(np.float32)   # (T, 2) = [yaw, pitch]
    np.save(str(out_path), gaze)
    return n_frames, len(valid_idx)

gaze_failures = []
todo = [v for v in cropped_videos if OVERWRITE or not (GAZE_DIR / (v.stem + ".npy")).exists()]
print(f"Extracting gaze for {len(todo)}/{len(cropped_videos)} videos ...")
for v in tqdm(todo, desc="gaze"):
    try:
        n, nv = extract_gaze(v, GAZE_DIR / (v.stem + ".npy"))
        if nv < n:
            print(f"   [warn] {v.name}: {n - nv}/{n} frames had no face - filled from nearest valid frame")
    except Exception as e:
        print(f"[FAIL] {v.name}: {e}")
        gaze_failures.append(v.name)

print(f"gaze/: {len(list(GAZE_DIR.glob('*.npy')))} files")



## Summary - what to do next


In [ ]:

def _count(d, ext):
    return len(list(d.glob("*" + ext)))

print("=" * 70)
print("VIDEO PREPROCESSING SUMMARY")
print("=" * 70)
print(f"raw videos           : {len(videos)}")
print(f"cropped 512x512      : {_count(CROPPED_VIDEO_DIR, '.mp4')}   (failures: {len(crop_failures)})")
print(f"motion/*.pt          : {_count(MOTION_DIR, '.pt')}    (failures: {len(motion_failures)})")
print(f"smirk/*.pt           : {_count(SMIRK_DIR, '.pt')}    (failures: {len(smirk_failures)})")
print(f"gaze/*.npy           : {_count(GAZE_DIR, '.npy')}    (failures: {len(gaze_failures)})")
print()
print("Next steps:")
print(f"  1. Run 02_preprocess_audio.ipynb  (fills {DATASET_DIR / 'audio'})")
print(f"  2. Run 03_verify_dataset.ipynb    (validates the whole dataset)")
print(f"  3. Train: python generator/train.py --dataset_pat \"{DATASET_DIR}\" --exp_name generator_exp --batch_size 16 --iter 5000000 --lr 1e-4")
